# PromptAlign: reviewed data → GPU extraction → regression

Read project_records/reports/results/G1_kaggle_full_qa_review.md before accepting the data decisions.
Set REVIEWER to your name and ACCEPT_DATA_REVIEW=True only after review.
Enable Internet and one Kaggle GPU. The repository scripts do the work.
Use the existing QA ZIP in /kaggle/working, or attach it as a private Kaggle Input.
The known dataset cache is reused; DOWNLOAD_DATASET=True explicitly permits
downloading the pinned ~4.76 GB snapshot if it is missing.
CLIP weights are downloaded to a revision-specific cache.
Only validation metrics are produced; test holdout is not evaluated.
Download training-*.zip through Output / Files, not an absolute-path hyperlink.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hoan25/BaiqiL-GenAI-Bench.git"
GIT_REF = "main"
PRIVATE_REPO = False
ROOT = Path("/kaggle/working/promptalign-ai")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
if PRIVATE_REPO:
    import base64
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authorization = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + authorization)
try:
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], env=env, check=True)
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=ROOT, text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Existing directory belongs to a different repository")
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip():
        raise RuntimeError("Working tree has changes; preserve them before changing revision")
    subprocess.run(["git", "fetch", "origin", GIT_REF], cwd=ROOT, env=env, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
finally:
    for key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
        env.pop(key, None)
    if PRIVATE_REPO:
        del token, authorization
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip())


In [ ]:
REVIEWER = ""  # name of the person reviewing the documented data decisions
ACCEPT_DATA_REVIEW = False
DOWNLOAD_DATASET = True  # explicit opt-in when executing; used only if cache is absent
QA_ARCHIVE = ""  # ZIP path OR unpacked root containing result.json and qa/
DATA_ROOT = ""  # optional existing folder with all 12 reviewed shards

if not REVIEWER.strip() or not ACCEPT_DATA_REVIEW:
    raise ValueError("Read the G1 data review, enter REVIEWER and set ACCEPT_DATA_REVIEW=True.")
if not QA_ARCHIVE:
    name = "data-qa-20261020T165712283760Z.zip"
    candidates = list(Path("/kaggle/working").glob(name))
    candidates += list(Path("/kaggle/input").rglob(name))
    if not candidates:
        candidates = [p.parent.parent for base in (Path("/kaggle/working"), Path("/kaggle/input"))
                      for p in base.rglob("qa/manifest.jsonl")
                      if (p.parent.parent / "dataset.json").is_file()]
    if len(candidates) != 1:
        raise ValueError("Set QA_ARCHIVE to the reviewed ZIP or its unpacked root directory.")
    QA_ARCHIVE = str(candidates[0])
command = [sys.executable, "-u", str(ROOT / "code/scripts/kaggle_train.py"),
           "--qa-archive", QA_ARCHIVE, "--reviewer", REVIEWER,
           "--accept-data-review", "--device", "cuda", "--batch-size", "16"]
if DATA_ROOT:
    command += ["--data-root", DATA_ROOT]
if DOWNLOAD_DATASET:
    command += ["--download-dataset"]
result = subprocess.run(command, cwd=ROOT / "code")
print("Exit code:", result.returncode)
print("Download the newest training-*.zip from Kaggle Output / Files:")
for archive in sorted(Path("/kaggle/working").glob("training-*.zip")):
    print(f"  {archive.name} ({archive.stat().st_size:,} bytes)")
